# CLIP4CAD-GFA Ablation Study Testing

This notebook tests and compares trained ablation checkpoints.

| Ablation | Description | Status |
|----------|-------------|--------|
| `baseline` | Full model | Reference |
| `no_consistency` | Disable grounding consistency | Tests grounding importance |
| `global_only` | Only global contrastive loss | Tests local alignment value |
| `no_confidence` | Fixed uniform confidence | Tests learned weighting |
| `weak_grounding` | Reduced grounding losses | Tests grounding strength |
| `asymmetric_grounding` | Asymmetric consistency weights | Tests modality-specific tuning |

## Features
- Load and evaluate trained ablation checkpoints
- Compare metrics across ablations
- Interactive text-to-CAD retrieval demo
- Gallery embedding caching for fast evaluation

## Usage
1. Run Cells 1-3 to set up and load test data
2. Run individual ablation test cells OR the comparison cell
3. Use text-to-CAD demo for interactive retrieval

---
## Cell 1: Imports & Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, 'D:/Defect_Det/MMCAD/MMCAD')

import gc
import json
import torch
import numpy as np
import pandas as pd
import h5py
import torch.nn.functional as F
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from omegaconf import OmegaConf

from clip4cad.ablations.models import CLIP4CAD_GFA_Ablation
from clip4cad.ablations.configs import get_ablation_config, ABLATION_TYPES
from clip4cad.data.gfa_dataset import GFAMappedDataset, gfa_collate_fn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Available ablation types:")
for name, desc in ABLATION_TYPES.items():
    print(f"  - {name}: {desc}")
print(f"\nDevice: {device}")
print("\n" + "="*60)
print("Run Cell 2 to configure paths, then Cell 3 to load test data.")
print("="*60)

---
## Cell 2: Configuration

In [ ]:
# =============================================================================
# PATHS - Update these for your setup
# =============================================================================
PATHS = {
    "data_root": "d:/Defect_Det/MMCAD/data",
    "pc_file": "c:/Users/User/Desktop/pc_embeddings_full.h5",
    "brep_file": "c:/Users/User/Desktop/brep_features.h5",
    "text_file": "c:/Users/User/Desktop/text_embeddings.h5",
    "config_path": "d:/Defect_Det/MMCAD/MMCAD/configs/model/clip4cad_gfa.yaml",
    "ablations_dir": "outputs/ablations",
    "metadata_csv": "d:/Defect_Det/MMCAD/data/169k.csv",
}

# =============================================================================
# DEFAULT CHECKPOINTS - Which checkpoint to use for each ablation
# =============================================================================
DEFAULT_CHECKPOINTS = {
    "baseline": "checkpoint_epoch35.pt",
    "no_consistency": "checkpoint_epoch35.pt",
    "global_only": "checkpoint_epoch35.pt",
    "no_confidence": "checkpoint_epoch35.pt",
    "weak_grounding": "checkpoint_epoch35.pt",
    "asymmetric_grounding": "checkpoint_epoch35.pt",
}

# =============================================================================
# EVALUATION CONFIG
# =============================================================================
EVAL_CONFIG = {
    "batch_size": 64,
    "num_workers": 0,  # 0 for memory-mapped data
    "k_values": [1, 5, 10],  # R@k and mAP@k
}

print("Configuration loaded.")
print(f"  Ablations dir: {PATHS['ablations_dir']}")
print(f"  Batch size: {EVAL_CONFIG['batch_size']}")
print(f"  K values: {EVAL_CONFIG['k_values']}")

---
## Cell 3: Load Test Dataset (one-time)

In [ ]:
print("Loading test dataset...")
print("="*60)

# Test dataset - ON DISK (saves RAM)
test_dataset = GFAMappedDataset(
    data_root=PATHS["data_root"],
    split="test",
    pc_file=PATHS["pc_file"],
    text_file=PATHS["text_file"],
    brep_file=PATHS["brep_file"],
    num_rotations=1,
    load_to_memory=False,
)
print(f"Test: {len(test_dataset):,} samples")

# Optionally load validation dataset
val_dataset = GFAMappedDataset(
    data_root=PATHS["data_root"],
    split="val",
    pc_file=PATHS["pc_file"],
    text_file=PATHS["text_file"],
    brep_file=PATHS["brep_file"],
    num_rotations=1,
    load_to_memory=False,
)
print(f"Val: {len(val_dataset):,} samples")

# Load metadata for display
try:
    metadata_df = pd.read_csv(PATHS["metadata_csv"]).set_index("uid")
    print(f"Metadata: {len(metadata_df):,} entries")
except Exception as e:
    print(f"Warning: Could not load metadata CSV: {e}")
    metadata_df = None

print("\n" + "="*60)
print("DATASET READY!")
print("="*60)

---
## Cell 4: Helper Functions

In [ ]:
# =============================================================================
# MODEL CACHE - Avoid reloading models
# =============================================================================
_model_cache = {}
_embedding_cache = {}


def load_ablation_model(ablation_type: str, checkpoint: str = None, force_reload: bool = False):
    """
    Load a trained ablation model from checkpoint.
    
    Args:
        ablation_type: e.g., "asymmetric_grounding"
        checkpoint: e.g., "checkpoint_epoch35.pt" or "best"
        force_reload: If True, reload even if cached
    
    Returns:
        model: CLIP4CAD_GFA_Ablation in eval mode
    """
    if checkpoint is None:
        checkpoint = DEFAULT_CHECKPOINTS.get(ablation_type, "checkpoint_epoch35.pt")
    
    cache_key = f"{ablation_type}_{checkpoint}"
    
    if cache_key in _model_cache and not force_reload:
        return _model_cache[cache_key]
    
    # Build checkpoint path
    output_dir = Path(PATHS["ablations_dir"]) / ablation_type
    
    if checkpoint == "best":
        ckpt_path = output_dir / "checkpoint_best.pt"
    else:
        ckpt_path = output_dir / checkpoint
    
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
    
    print(f"Loading {ablation_type} from {ckpt_path.name}...")
    
    # Load checkpoint
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    
    # Get ablation config
    config = get_ablation_config(PATHS["config_path"], ablation_type)
    
    # Create and load model
    # Use strict=False for backward compatibility with older checkpoints
    # (e.g., checkpoints trained before align_text was added)
    model = CLIP4CAD_GFA_Ablation(config)
    missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=False)
    
    if missing:
        print(f"  Note: Missing keys (will use random init): {len(missing)} keys")
        # Only show first few if many
        for key in missing[:3]:
            print(f"    - {key}")
        if len(missing) > 3:
            print(f"    ... and {len(missing) - 3} more")
    
    if unexpected:
        print(f"  Warning: Unexpected keys in checkpoint: {unexpected}")
    
    model = model.to(device).eval()
    
    # Cache
    _model_cache[cache_key] = model
    
    # Print info
    epoch = ckpt.get("epoch", "?")
    stage = ckpt.get("stage", "?")
    print(f"  Loaded epoch {epoch}, stage {stage}")
    
    return model


def compute_gallery_embeddings(model, dataset, batch_size: int = 64, force_recompute: bool = False):
    """
    Compute and cache gallery embeddings for a model.
    
    Uses self-grounding (no text required) for geometry embeddings.
    
    Args:
        model: CLIP4CAD_GFA_Ablation in eval mode
        dataset: GFAMappedDataset
        batch_size: Batch size for embedding computation
        force_recompute: If True, recompute even if cached
    
    Returns:
        embeddings: Dict with 'text', 'brep', 'pc' embeddings (B, d)
        sample_ids: List of sample IDs
    """
    # Cache key includes model identity
    cache_key = f"{id(model)}_{len(dataset)}"
    
    if cache_key in _embedding_cache and not force_recompute:
        return _embedding_cache[cache_key]
    
    print(f"Computing gallery embeddings for {len(dataset):,} samples...")
    
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=True, collate_fn=gfa_collate_fn
    )
    
    text_emb, brep_emb, pc_emb = [], [], []
    sample_ids = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="  Embeddings", leave=False):
            sample_ids.extend(batch["sample_id"])
            
            # Forward pass
            outputs = model(batch)
            
            # Normalize and store
            text_emb.append(F.normalize(outputs["z_text"].float(), p=2, dim=-1).cpu())
            
            if "z_brep" in outputs:
                brep_emb.append(F.normalize(outputs["z_brep"].float(), p=2, dim=-1).cpu())
            
            if "z_pc" in outputs:
                pc_emb.append(F.normalize(outputs["z_pc"].float(), p=2, dim=-1).cpu())
    
    embeddings = {
        "text": torch.cat(text_emb),
        "brep": torch.cat(brep_emb) if brep_emb else None,
        "pc": torch.cat(pc_emb) if pc_emb else None,
    }
    
    # Cache
    _embedding_cache[cache_key] = (embeddings, sample_ids)
    
    print(f"  Cached {len(sample_ids):,} embeddings")
    
    return embeddings, sample_ids


def compute_retrieval_metrics(query_emb, gallery_emb, sample_ids, k_values=[1, 5, 10]):
    """
    Compute retrieval metrics (R@k, mAP@k).
    
    Args:
        query_emb: Query embeddings (N, d)
        gallery_emb: Gallery embeddings (N, d)
        sample_ids: List of sample IDs (for identity matching)
        k_values: List of k values for metrics
    
    Returns:
        metrics: Dict with R@k and mAP@k values
    """
    # Compute similarities
    sim = torch.mm(query_emb, gallery_emb.T)
    _, rankings = torch.topk(sim, k=max(k_values), dim=1)
    
    n = len(sample_ids)
    metrics = {}
    
    for k in k_values:
        hits = 0
        ap_sum = 0.0
        
        for i in range(n):
            qid = str(sample_ids[i])
            for rank, idx in enumerate(rankings[i, :k]):
                if str(sample_ids[idx]) == qid:
                    hits += 1
                    ap_sum += 1.0 / (rank + 1)
                    break
        
        metrics[f"R@{k}"] = hits / n
        metrics[f"mAP@{k}"] = ap_sum / n
    
    return metrics


def evaluate_ablation(ablation_type: str, dataset, checkpoint: str = None):
    """
    Evaluate an ablation on a dataset.
    
    Args:
        ablation_type: e.g., "asymmetric_grounding"
        dataset: GFAMappedDataset (test or val)
        checkpoint: Optional checkpoint name
    
    Returns:
        results: Dict with metrics for all retrieval tasks
    """
    # Load model
    model = load_ablation_model(ablation_type, checkpoint)
    
    # Compute embeddings
    embeddings, sample_ids = compute_gallery_embeddings(
        model, dataset, batch_size=EVAL_CONFIG["batch_size"]
    )
    
    k_values = EVAL_CONFIG["k_values"]
    results = {}
    
    # Text -> BRep
    if embeddings["brep"] is not None:
        results["Text->BRep"] = compute_retrieval_metrics(
            embeddings["text"], embeddings["brep"], sample_ids, k_values
        )
    
    # Text -> PC
    if embeddings["pc"] is not None:
        results["Text->PC"] = compute_retrieval_metrics(
            embeddings["text"], embeddings["pc"], sample_ids, k_values
        )
    
    # PC -> BRep
    if embeddings["brep"] is not None and embeddings["pc"] is not None:
        results["PC->BRep"] = compute_retrieval_metrics(
            embeddings["pc"], embeddings["brep"], sample_ids, k_values
        )
    
    # BRep -> PC
    if embeddings["brep"] is not None and embeddings["pc"] is not None:
        results["BRep->PC"] = compute_retrieval_metrics(
            embeddings["brep"], embeddings["pc"], sample_ids, k_values
        )
    
    return results


def retrieve_top_k(query_emb, gallery_emb, gallery_ids, k: int = 10):
    """
    Retrieve top-k most similar items.
    
    Args:
        query_emb: Query embedding (1, d) or (d,)
        gallery_emb: Gallery embeddings (N, d)
        gallery_ids: List of gallery IDs
        k: Number of results
    
    Returns:
        results: List of (uid, score) tuples
    """
    if query_emb.dim() == 1:
        query_emb = query_emb.unsqueeze(0)
    
    # Compute similarities
    sim = torch.mm(query_emb, gallery_emb.T).squeeze(0)  # (N,)
    
    # Get top-k
    scores, indices = torch.topk(sim, k=min(k, len(gallery_ids)))
    
    results = [
        (gallery_ids[idx], scores[i].item())
        for i, idx in enumerate(indices)
    ]
    
    return results


def display_results(results, metadata_df=None, show_scores: bool = True):
    """
    Display retrieval results with metadata.
    
    Args:
        results: List of (uid, score) tuples
        metadata_df: Optional DataFrame with metadata
        show_scores: Whether to show similarity scores
    """
    print(f"\n{'Rank':<6} {'UID':<40} {'Score':<8} {'Title'}")
    print("-" * 100)
    
    for rank, (uid, score) in enumerate(results, 1):
        title = "N/A"
        if metadata_df is not None and uid in metadata_df.index:
            title = str(metadata_df.loc[uid, "title"])[:50]
        
        score_str = f"{score:.4f}" if show_scores else ""
        print(f"{rank:<6} {uid:<40} {score_str:<8} {title}")


def clear_cache():
    """Clear model and embedding caches."""
    global _model_cache, _embedding_cache
    _model_cache.clear()
    _embedding_cache.clear()
    gc.collect()
    torch.cuda.empty_cache()
    print("Cache cleared.")


print("Helper functions defined.")
print("  - load_ablation_model(ablation_type, checkpoint)")
print("  - compute_gallery_embeddings(model, dataset)")
print("  - evaluate_ablation(ablation_type, dataset, checkpoint)")
print("  - retrieve_top_k(query_emb, gallery_emb, gallery_ids, k)")
print("  - display_results(results, metadata_df)")
print("  - clear_cache()")

---
## Cell 5: Quick Evaluation Function

In [ ]:
def test_ablation(ablation_type: str, checkpoint: str = None, split: str = "test"):
    """
    Quick evaluation of a single ablation.
    
    Args:
        ablation_type: e.g., "asymmetric_grounding"
        checkpoint: e.g., "checkpoint_epoch35.pt" or "best"
        split: "val" or "test"
    
    Returns:
        model: Loaded model (for further use)
        results: Dict with metrics
    """
    print(f"\n{'='*70}")
    print(f"TESTING: {ablation_type}")
    print(f"Checkpoint: {checkpoint or DEFAULT_CHECKPOINTS.get(ablation_type, 'best')}")
    print(f"Split: {split}")
    print(f"{'='*70}")
    
    # Select dataset
    dataset = test_dataset if split == "test" else val_dataset
    
    # Load model
    model = load_ablation_model(ablation_type, checkpoint)
    
    # Evaluate
    results = evaluate_ablation(ablation_type, dataset, checkpoint)
    
    # Print results
    print("\nResults:")
    for task, metrics in results.items():
        if isinstance(metrics, dict):
            r1 = metrics.get('R@1', 0)
            r5 = metrics.get('R@5', 0)
            r10 = metrics.get('R@10', 0)
            map1 = metrics.get('mAP@1', 0)
            print(f"  {task}: R@1={r1:.4f}, R@5={r5:.4f}, R@10={r10:.4f}, mAP@1={map1:.4f}")
    
    return model, results


print("test_ablation() function defined.")
print("Usage: model, results = test_ablation('asymmetric_grounding')")

---
---
# INDIVIDUAL ABLATION TEST CELLS

Run ONE of the cells below to test that specific ablation.

---

## Test: ASYMMETRIC GROUNDING

In [ ]:
# === TEST: ASYMMETRIC GROUNDING ===
model, results = test_ablation("asymmetric_grounding", checkpoint="checkpoint_epoch35.pt")

## Test: WEAK GROUNDING

In [ ]:
# === TEST: WEAK GROUNDING ===
model, results = test_ablation("weak_grounding")

## Test: BASELINE

In [ ]:
# === TEST: BASELINE ===
model, results = test_ablation("baseline")

## Test: NO CONSISTENCY

In [ ]:
# === TEST: NO CONSISTENCY ===
model, results = test_ablation("no_consistency")

## Test: GLOBAL ONLY

In [ ]:
# === TEST: GLOBAL ONLY ===
model, results = test_ablation("global_only")

## Test: NO CONFIDENCE

In [ ]:
# === TEST: NO CONFIDENCE ===
model, results = test_ablation("no_confidence")

---
---
# SELF-GROUNDING EVALUATION

After Phase 2 training, compare text-grounded vs self-grounded embeddings.

**Good alignment indicators:**
- Cosine similarity > 0.95
- Standard deviation < 0.05

---

## Self-Grounding Validation Functions

In [ ]:
# =============================================================================
# SELF-GROUNDING EVALUATION FUNCTIONS
# =============================================================================

def validate_self_grounding_alignment(ablation_type: str, checkpoint: str = None,
                                       dataset=None, batch_size: int = 64):
    """
    Validate that self-grounded embeddings match text-grounded embeddings.
    
    Args:
        ablation_type: e.g., "asymmetric_grounding"
        checkpoint: Checkpoint name (default: checkpoint_phase2_final.pt)
        dataset: Dataset to use (defaults to test_dataset)
        batch_size: Batch size for computation
    
    Returns:
        metrics: Dict with cosine similarity stats
    """
    if dataset is None:
        dataset = test_dataset
    
    if checkpoint is None:
        checkpoint = "checkpoint_phase2_final.pt"
    
    print(f"\n{'='*70}")
    print(f"SELF-GROUNDING ALIGNMENT: {ablation_type}")
    print(f"Checkpoint: {checkpoint}")
    print(f"{'='*70}")
    
    # Load model
    model = load_ablation_model(ablation_type, checkpoint)
    
    # Create dataloader
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=True, collate_fn=gfa_collate_fn
    )
    
    cosine_sims_brep = []
    cosine_sims_pc = []
    
    print("Computing embeddings...")
    with torch.no_grad():
        for batch in tqdm(loader, desc="  Processing", leave=False):
            # Get text-grounded embeddings
            text_grounded = model.compute_text_grounded_embeddings(batch)
            
            # Get self-grounded embeddings
            self_grounded = model.compute_self_grounded_embeddings(batch)
            
            # B-Rep similarity
            if "z_brep" in text_grounded and "z_brep" in self_grounded:
                sim_brep = F.cosine_similarity(
                    text_grounded["z_brep"], self_grounded["z_brep"], dim=-1
                )
                cosine_sims_brep.extend(sim_brep.cpu().tolist())
            
            # PC similarity
            if "z_pc" in text_grounded and "z_pc" in self_grounded:
                sim_pc = F.cosine_similarity(
                    text_grounded["z_pc"], self_grounded["z_pc"], dim=-1
                )
                cosine_sims_pc.extend(sim_pc.cpu().tolist())
    
    # Compute statistics
    metrics = {}
    
    if cosine_sims_brep:
        metrics["brep_mean"] = np.mean(cosine_sims_brep)
        metrics["brep_std"] = np.std(cosine_sims_brep)
        metrics["brep_min"] = np.min(cosine_sims_brep)
        metrics["brep_max"] = np.max(cosine_sims_brep)
    
    if cosine_sims_pc:
        metrics["pc_mean"] = np.mean(cosine_sims_pc)
        metrics["pc_std"] = np.std(cosine_sims_pc)
        metrics["pc_min"] = np.min(cosine_sims_pc)
        metrics["pc_max"] = np.max(cosine_sims_pc)
    
    # Print results
    print("\nSelf-Grounding Alignment Results:")
    print("-" * 50)
    
    if "brep_mean" in metrics:
        quality = "GOOD" if metrics["brep_mean"] > 0.95 else "NEEDS IMPROVEMENT"
        print(f"  B-Rep: mean={metrics['brep_mean']:.4f} ± {metrics['brep_std']:.4f} [{quality}]")
        print(f"         range=[{metrics['brep_min']:.4f}, {metrics['brep_max']:.4f}]")
    
    if "pc_mean" in metrics:
        quality = "GOOD" if metrics["pc_mean"] > 0.95 else "NEEDS IMPROVEMENT"
        print(f"  PC:    mean={metrics['pc_mean']:.4f} ± {metrics['pc_std']:.4f} [{quality}]")
        print(f"         range=[{metrics['pc_min']:.4f}, {metrics['pc_max']:.4f}]")
    
    return metrics


def compare_grounding_methods(ablation_type: str, checkpoint: str = None,
                               dataset=None, batch_size: int = 64):
    """
    Compare retrieval using text-grounded vs self-grounded embeddings.
    
    Args:
        ablation_type: e.g., "asymmetric_grounding"
        checkpoint: Checkpoint name (default: checkpoint_phase2_final.pt)
        dataset: Dataset to use (defaults to test_dataset)
        batch_size: Batch size for computation
    
    Returns:
        results: Dict with metrics for both grounding methods
    """
    if dataset is None:
        dataset = test_dataset
    
    if checkpoint is None:
        checkpoint = "checkpoint_phase2_final.pt"
    
    print(f"\n{'='*70}")
    print(f"GROUNDING METHOD COMPARISON: {ablation_type}")
    print(f"{'='*70}")
    
    # Load model
    model = load_ablation_model(ablation_type, checkpoint)
    
    # Create dataloader
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=True, collate_fn=gfa_collate_fn
    )
    
    # Compute both types of embeddings
    text_grounded_brep = []
    text_grounded_pc = []
    self_grounded_brep = []
    self_grounded_pc = []
    text_emb = []
    sample_ids = []
    
    print("Computing embeddings...")
    with torch.no_grad():
        for batch in tqdm(loader, desc="  Processing", leave=False):
            sample_ids.extend(batch["sample_id"])
            
            # Text embeddings (for text->geometry retrieval)
            outputs = model(batch)
            text_emb.append(F.normalize(outputs["z_text"].float(), p=2, dim=-1).cpu())
            
            # Text-grounded geometry embeddings
            tg = model.compute_text_grounded_embeddings(batch)
            if "z_brep" in tg:
                text_grounded_brep.append(F.normalize(tg["z_brep"].float(), p=2, dim=-1).cpu())
            if "z_pc" in tg:
                text_grounded_pc.append(F.normalize(tg["z_pc"].float(), p=2, dim=-1).cpu())
            
            # Self-grounded geometry embeddings
            sg = model.compute_self_grounded_embeddings(batch)
            if "z_brep" in sg:
                self_grounded_brep.append(F.normalize(sg["z_brep"].float(), p=2, dim=-1).cpu())
            if "z_pc" in sg:
                self_grounded_pc.append(F.normalize(sg["z_pc"].float(), p=2, dim=-1).cpu())
    
    # Concatenate
    text_emb = torch.cat(text_emb)
    text_grounded_brep = torch.cat(text_grounded_brep) if text_grounded_brep else None
    text_grounded_pc = torch.cat(text_grounded_pc) if text_grounded_pc else None
    self_grounded_brep = torch.cat(self_grounded_brep) if self_grounded_brep else None
    self_grounded_pc = torch.cat(self_grounded_pc) if self_grounded_pc else None
    
    k_values = EVAL_CONFIG["k_values"]
    results = {"text_grounded": {}, "self_grounded": {}}
    
    # Text -> BRep with text-grounded embeddings
    if text_grounded_brep is not None:
        results["text_grounded"]["Text->BRep"] = compute_retrieval_metrics(
            text_emb, text_grounded_brep, sample_ids, k_values
        )
    
    # Text -> BRep with self-grounded embeddings
    if self_grounded_brep is not None:
        results["self_grounded"]["Text->BRep"] = compute_retrieval_metrics(
            text_emb, self_grounded_brep, sample_ids, k_values
        )
    
    # Text -> PC with text-grounded embeddings
    if text_grounded_pc is not None:
        results["text_grounded"]["Text->PC"] = compute_retrieval_metrics(
            text_emb, text_grounded_pc, sample_ids, k_values
        )
    
    # Text -> PC with self-grounded embeddings
    if self_grounded_pc is not None:
        results["self_grounded"]["Text->PC"] = compute_retrieval_metrics(
            text_emb, self_grounded_pc, sample_ids, k_values
        )
    
    # Print comparison table
    print("\nText->BRep Retrieval Comparison:")
    print("-" * 60)
    print(f"{'Method':<20} {'R@1':>10} {'R@5':>10} {'R@10':>10}")
    print("-" * 60)
    
    for method in ["text_grounded", "self_grounded"]:
        if "Text->BRep" in results[method]:
            m = results[method]["Text->BRep"]
            print(f"{method:<20} {m['R@1']:>10.4f} {m['R@5']:>10.4f} {m['R@10']:>10.4f}")
    
    print("\nText->PC Retrieval Comparison:")
    print("-" * 60)
    print(f"{'Method':<20} {'R@1':>10} {'R@5':>10} {'R@10':>10}")
    print("-" * 60)
    
    for method in ["text_grounded", "self_grounded"]:
        if "Text->PC" in results[method]:
            m = results[method]["Text->PC"]
            print(f"{method:<20} {m['R@1']:>10.4f} {m['R@5']:>10.4f} {m['R@10']:>10.4f}")
    
    return results


print("Self-grounding evaluation functions defined:")
print("  - validate_self_grounding_alignment(ablation_type, checkpoint)")
print("  - compare_grounding_methods(ablation_type, checkpoint)")

## Validate Self-Grounding: ASYMMETRIC GROUNDING

Check cosine similarity between text-grounded and self-grounded embeddings.

In [ ]:
# === VALIDATE SELF-GROUNDING: ASYMMETRIC GROUNDING ===
# Prerequisites: Phase 2 training must be complete (checkpoint_phase2_final.pt exists)
metrics = validate_self_grounding_alignment("asymmetric_grounding")

## Compare Grounding Methods: ASYMMETRIC GROUNDING

Compare retrieval quality using text-grounded vs self-grounded embeddings.

In [ ]:
# === COMPARE GROUNDING METHODS: ASYMMETRIC GROUNDING ===
results = compare_grounding_methods("asymmetric_grounding")

## Validate & Compare: WEAK GROUNDING

In [ ]:
# === VALIDATE & COMPARE: WEAK GROUNDING ===
metrics_wg = validate_self_grounding_alignment("weak_grounding")
results_wg = compare_grounding_methods("weak_grounding")

## Compare All Self-Grounding Models

Compare self-grounding alignment quality across all ablations with Phase 2 checkpoints.

In [ ]:
# === COMPARE ALL SELF-GROUNDING MODELS ===

def compare_all_self_grounding(ablation_list: list, dataset=None):
    """
    Compare self-grounding quality across multiple ablations.
    """
    if dataset is None:
        dataset = test_dataset
    
    print("=" * 80)
    print("SELF-GROUNDING ALIGNMENT COMPARISON (Phase 2)")
    print("=" * 80)
    
    all_metrics = {}
    
    for abl in ablation_list:
        # Check if Phase 2 checkpoint exists
        ckpt_path = Path(PATHS["ablations_dir"]) / abl / "checkpoint_phase2_final.pt"
        if not ckpt_path.exists():
            print(f"\n{abl}: Phase 2 checkpoint not found, skipping...")
            continue
        
        print(f"\nProcessing {abl}...")
        try:
            metrics = validate_self_grounding_alignment(abl, dataset=dataset)
            all_metrics[abl] = metrics
        except Exception as e:
            print(f"  Error: {e}")
            continue
    
    # Print summary table
    print("\n" + "=" * 80)
    print("SUMMARY: Self-Grounding Alignment")
    print("=" * 80)
    print(f"{'Ablation':<25} {'B-Rep Mean':>12} {'B-Rep Std':>12} {'PC Mean':>12} {'PC Std':>12}")
    print("-" * 80)
    
    for abl, m in all_metrics.items():
        brep_mean = m.get("brep_mean", 0)
        brep_std = m.get("brep_std", 0)
        pc_mean = m.get("pc_mean", 0)
        pc_std = m.get("pc_std", 0)
        print(f"{abl:<25} {brep_mean:>12.4f} {brep_std:>12.4f} {pc_mean:>12.4f} {pc_std:>12.4f}")
    
    print("=" * 80)
    print("Target: mean > 0.95, std < 0.05")
    
    return all_metrics


# Compare all ablations with Phase 2 checkpoints
ablations_to_compare = [
    "asymmetric_grounding",
    "weak_grounding",
    # "baseline",
    # "no_consistency",
    # "global_only",
    # "no_confidence",
]

all_sg_metrics = compare_all_self_grounding(ablations_to_compare)

---
---
# COMPARISON & ANALYSIS

---

## Compare All Ablations

In [ ]:
def compare_ablations(ablation_list: list, checkpoint: str = None, split: str = "test"):
    """
    Compare multiple ablations side-by-side.
    
    Args:
        ablation_list: List of ablation types to compare
        checkpoint: Checkpoint to use (or None for defaults)
        split: "val" or "test"
    
    Returns:
        all_results: Dict mapping ablation -> results
    """
    dataset = test_dataset if split == "test" else val_dataset
    
    all_results = {}
    
    for abl in ablation_list:
        print(f"\nEvaluating {abl}...")
        try:
            results = evaluate_ablation(abl, dataset, checkpoint)
            all_results[abl] = results
        except FileNotFoundError as e:
            print(f"  Skipping {abl}: {e}")
            continue
    
    # Print comparison table
    print("\n" + "="*100)
    print(f"COMPARISON TABLE ({split} split)")
    print("="*100)
    
    # Header
    print(f"{'Ablation':<25} | {'Text->BRep':^30} | {'Text->PC':^30}")
    print(f"{'':25} | {'R@1':>8} {'R@5':>8} {'R@10':>8} | {'R@1':>8} {'R@5':>8} {'R@10':>8}")
    print("-"*100)
    
    for abl, res in all_results.items():
        tb = res.get("Text->BRep", {})
        tp = res.get("Text->PC", {})
        
        print(f"{abl:<25} | "
              f"{tb.get('R@1', 0):>8.4f} {tb.get('R@5', 0):>8.4f} {tb.get('R@10', 0):>8.4f} | "
              f"{tp.get('R@1', 0):>8.4f} {tp.get('R@5', 0):>8.4f} {tp.get('R@10', 0):>8.4f}")
    
    print("="*100)
    
    # Cross-modal comparison
    print(f"\n{'Ablation':<25} | {'PC->BRep':^30} | {'BRep->PC':^30}")
    print(f"{'':25} | {'R@1':>8} {'R@5':>8} {'R@10':>8} | {'R@1':>8} {'R@5':>8} {'R@10':>8}")
    print("-"*100)
    
    for abl, res in all_results.items():
        pb = res.get("PC->BRep", {})
        bp = res.get("BRep->PC", {})
        
        print(f"{abl:<25} | "
              f"{pb.get('R@1', 0):>8.4f} {pb.get('R@5', 0):>8.4f} {pb.get('R@10', 0):>8.4f} | "
              f"{bp.get('R@1', 0):>8.4f} {bp.get('R@5', 0):>8.4f} {bp.get('R@10', 0):>8.4f}")
    
    print("="*100)
    
    return all_results


# Compare trained ablations
# Uncomment the ablations you have trained
ablations_to_compare = [
    "asymmetric_grounding",
    "weak_grounding",
    # "baseline",
    # "no_consistency",
    # "global_only",
    # "no_confidence",
]

all_results = compare_ablations(ablations_to_compare, split="val")

## Save Comparison Results

In [ ]:
# Save comparison results to JSON
if 'all_results' in dir() and all_results:
    results_path = Path(PATHS["ablations_dir"]) / "ablation_comparison.json"
    with open(results_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"Results saved to: {results_path}")
else:
    print("Run the comparison cell first!")

---
---
# TEXT-TO-CAD RETRIEVAL DEMO

---

## Text-to-CAD Search Function

In [ ]:
def text_to_cad(query: str, ablation_type: str = "asymmetric_grounding",
                modality: str = "brep", k: int = 10, dataset=None):
    """
    Search CAD library with natural language query.
    
    NOTE: This uses pre-computed text embeddings from the dataset,
    not live text encoding. For arbitrary queries, you would need
    to integrate with the text encoder (LLM).
    
    This demo finds the closest text in the dataset to your query
    and retrieves the corresponding geometry.
    
    Args:
        query: Text description (for display)
        ablation_type: Which ablation model to use
        modality: "brep" or "pc"
        k: Number of results
        dataset: Dataset to use (defaults to test_dataset)
    
    Returns:
        results: List of (uid, score) tuples
    """
    if dataset is None:
        dataset = test_dataset
    
    print(f"\nQuery: '{query}'")
    print(f"Model: {ablation_type}")
    print(f"Modality: {modality}")
    print("-" * 60)
    
    # Load model
    model = load_ablation_model(ablation_type)
    
    # Compute embeddings
    embeddings, sample_ids = compute_gallery_embeddings(model, dataset)
    
    # For this demo, we use a random text embedding as the "query"
    # In practice, you would encode the query text using the model's text encoder
    print(f"\nNote: Using pre-computed text embeddings (live encoding not implemented)")
    print(f"Showing {modality}->{modality} self-retrieval for demonstration.\n")
    
    # Use first sample's embedding as query
    gallery_emb = embeddings[modality]
    query_emb = gallery_emb[0:1]  # First sample
    
    results = retrieve_top_k(query_emb, gallery_emb, sample_ids, k)
    display_results(results, metadata_df)
    
    return results


print("text_to_cad() function defined.")
print("Note: Requires live text encoding for arbitrary queries.")
print("Current implementation uses pre-computed embeddings.")

## Interactive Text-to-CAD Demo

In [ ]:
# Demo: Show retrieval results
# These use pre-computed embeddings - actual query is for display only

text_to_cad("hexagonal bolt with threaded shaft", ablation_type="asymmetric_grounding", modality="brep")

In [ ]:
text_to_cad("L-shaped mounting bracket with holes", ablation_type="weak_grounding", modality="pc")

## Compare Retrieval Across Ablations

In [ ]:
def compare_retrieval(query_idx: int, ablations: list, modality: str = "brep", k: int = 5):
    """
    Compare retrieval results for same query across ablations.
    
    Args:
        query_idx: Index of sample to use as query
        ablations: List of ablation types to compare
        modality: "brep" or "pc"
        k: Number of results per ablation
    """
    print(f"Query: Sample #{query_idx}")
    print(f"Modality: {modality}")
    print("="*80)
    
    for abl in ablations:
        print(f"\n--- {abl} ---")
        
        try:
            model = load_ablation_model(abl)
            embeddings, sample_ids = compute_gallery_embeddings(model, test_dataset)
            
            gallery_emb = embeddings[modality]
            query_emb = gallery_emb[query_idx:query_idx+1]
            
            results = retrieve_top_k(query_emb, gallery_emb, sample_ids, k)
            
            for rank, (uid, score) in enumerate(results[:k], 1):
                title = "N/A"
                if metadata_df is not None and uid in metadata_df.index:
                    title = str(metadata_df.loc[uid, "title"])[:40]
                print(f"  {rank}. {uid[:20]}... ({score:.3f}): {title}")
        
        except FileNotFoundError:
            print(f"  [Checkpoint not found]")


# Compare retrieval for sample #0 across ablations
compare_retrieval(0, ["asymmetric_grounding", "weak_grounding"], modality="brep", k=5)

---
---
# UTILITIES

---

## Memory Check

In [ ]:
import psutil

gc.collect()
torch.cuda.empty_cache()

process = psutil.Process()
ram_gb = process.memory_info().rss / 1e9
available_ram = psutil.virtual_memory().available / 1e9

print(f"Process RAM: {ram_gb:.1f} GB")
print(f"Available RAM: {available_ram:.1f} GB")

if torch.cuda.is_available():
    gpu_alloc = torch.cuda.memory_allocated() / 1e9
    gpu_cached = torch.cuda.memory_reserved() / 1e9
    print(f"GPU Allocated: {gpu_alloc:.1f} GB")
    print(f"GPU Cached: {gpu_cached:.1f} GB")

print(f"\nCached models: {len(_model_cache)}")
print(f"Cached embeddings: {len(_embedding_cache)}")

## Clear Caches

In [ ]:
# Run this cell to clear caches and free memory
clear_cache()

## List Available Checkpoints

In [ ]:
def list_checkpoints():
    """List all available ablation checkpoints."""
    ablations_dir = Path(PATHS["ablations_dir"])
    
    print("Available checkpoints:")
    print("="*60)
    
    for abl_dir in sorted(ablations_dir.iterdir()):
        if abl_dir.is_dir():
            checkpoints = list(abl_dir.glob("checkpoint_*.pt"))
            if checkpoints:
                print(f"\n{abl_dir.name}:")
                for ckpt in sorted(checkpoints):
                    size_mb = ckpt.stat().st_size / 1e6
                    print(f"  - {ckpt.name} ({size_mb:.1f} MB)")

list_checkpoints()